In [1]:
import pandas as pd

# 1. Load the flat CSV file
# Replace 'temp.csv' with your actual file path if needed
df = pd.read_csv('data/transformed_data.csv')

print("Starting data transformation...")

# ==========================================
# TABLE 1: Source
# ==========================================
# Extract unique sources and assign a SourceID (e.g., S01, S02)
sources_df = df[['source']].drop_duplicates().reset_index(drop=True)
sources_df['SourceID'] = ['S' + str(i+1).zfill(2) for i in range(len(sources_df))]

# Rename to match schema and reorder
sources_df = sources_df.rename(columns={'source': 'SourceName'})
sources_df = sources_df[['SourceID', 'SourceName']]

# Create a dictionary mapping for later (e.g., {'An Phat PC': 'S01'})
source_map = dict(zip(sources_df['SourceName'], sources_df['SourceID']))

# ==========================================
# TABLE 2: Product
# ==========================================
# Define the columns that belong to the Product table
product_cols_input = [
    'name', 'brand', 'category', 'cpu_brand', 'cpu_family', 'gpu_type', 'gpu_model',
    'ram_gb', 'ram_type', 'storage_gb', 'storage_type', 'screen_size_inches',
    'screen_res', 'screen_hz', 'os_family', 'weight_kg', 'is_ai', 'is_available'
]

# Extract unique products and assign a ProductID (e.g., P0001, P0002)
products_df = df[product_cols_input].drop_duplicates().reset_index(drop=True)
products_df['ProductID'] = ['P' + str(i+1).zfill(4) for i in range(len(products_df))]

# Rename columns to match your exact schema image
# Note: I included 'Brand' as it is vital for your analytics, even if omitted in the image text
products_df = products_df.rename(columns={
    'name': 'ProductName',
    'brand': 'Brand', 
    'category': 'Category',
    'cpu_brand': 'CPUBrand',
    'cpu_family': 'CPUFamily',
    'gpu_type': 'GPUType',
    'gpu_model': 'GPUModel',
    'ram_gb': 'RamCapacity',
    'ram_type': 'RamType',
    'storage_gb': 'StorageCapacity',
    'storage_type': 'StorageType',
    'screen_size_inches': 'DisplaySize',
    'screen_res': 'DisplayResolution',
    'screen_hz': 'DisplayRefreshRate',
    'os_family': 'OSFamily',
    'weight_kg': 'DesignWeight',
    'is_ai': 'IsAiLaptop',
    'is_available': 'IsAvailable'
})

# Reorder columns to put ProductID first
product_cols_final = ['ProductID'] + [c for c in products_df.columns if c != 'ProductID']
products_df = products_df[product_cols_final]

# ==========================================
# TABLE 3: Source_Product (The Fact Table)
# ==========================================
# Create a copy of the original dataframe
sp_df = df.copy()

# 1. Map the SourceID
sp_df['SourceID'] = sp_df['source'].map(source_map)

# 2. Map the ProductID 
# We join based on the exact product name to link the generated ProductID back to the main table
sp_df = sp_df.merge(products_df[['ProductID', 'ProductName']], 
                    left_on='name', 
                    right_on='ProductName', 
                    how='left')

# 3. Select only the columns needed for the bridge table
source_product_df = sp_df[[
    'SourceID', 'ProductID', 'avg_rating', 'review_count', 'satisfied_count', 'price'
]]

# 4. Rename columns to match your schema image
source_product_df = source_product_df.rename(columns={
    'avg_rating': 'AvgRating',
    'review_count': 'ReviewCount',
    'satisfied_count': 'SatisfiedCount',
    'price': 'CurrentPrice'
})

# ==========================================
# EXPORT TO CSV
# ==========================================
sources_df.to_csv('Dim_Source.csv', index=False, encoding='utf-8-sig')
products_df.to_csv('Dim_Product.csv', index=False, encoding='utf-8-sig')
source_product_df.to_csv('Fact_Source_Product.csv', index=False, encoding='utf-8-sig')

print("✅ Transformation Complete!")
print(f"Generated Dim_Source.csv with {len(sources_df)} rows.")
print(f"Generated Dim_Product.csv with {len(products_df)} rows.")
print(f"Generated Fact_Source_Product.csv with {len(source_product_df)} rows.")

Starting data transformation...
✅ Transformation Complete!
Generated Dim_Source.csv with 6 rows.
Generated Dim_Product.csv with 2764 rows.
Generated Fact_Source_Product.csv with 2817 rows.
